In [ ]:
#@title Bölüm sesini üret (T4 GPU) — çalıştır, bundle.<dil>.zip'i seç, bitince indirilen zip'i Mac'te unpack et
import glob, json, os, subprocess, time, zipfile
from google.colab import files
t0 = time.time()
assert subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode == 0, "GPU yok: Runtime → Change runtime type → T4 GPU"
if not os.path.exists("/content/.chatterbox_ok"):  # oturum başına bir kez (~4 dk)
    !pip install -q chatterbox-tts==0.1.7 2>&1 | grep -v "dependency resolver\|requires\|^$" || true
    !pip uninstall -y -q torchvision  # Colab'ın torchvision'ı torch 2.6 ile uyuşmuyor (torchvision::nms)
    open("/content/.chatterbox_ok", "w").close()
print(f"[kurulum] {time.time() - t0:.0f}s")

!rm -rf /content/cs && mkdir -p /content/cs
os.chdir("/content/cs")
for name, data in files.upload().items():
    zipfile.ZipFile(name).extractall("/content/cs")
    os.remove(name)
m = json.load(open("colab_manifest.json"))
print("[bölüm]", m["episode"], m["lang"], m["extra_args"])

extra = " ".join(m["extra_args"])
!python -W ignore voice/synth.py {m["script"]} --device cuda --no-asr {extra} 2>&1 | grep -E "^\[|Error|error"

zp = f"/content/voice_{m['episode']}.{m['lang']}.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(m["out"] + "/**", recursive=True):
        if os.path.isfile(f) and not f.endswith("narration.wav"):  # anlatım Mac'te gate sonrası yazılır
            z.write(f)
    z.write("colab_manifest.json")
print(f"[toplam] {time.time() - t0:.0f}s — indiriliyor: {zp}")
files.download(zp)
print("Bitti. Kota boşa gitmesin: Runtime → Disconnect and delete runtime")
